# Assignment in short
We have a npy file with all the data. 3 columns, User ID, Movie ID, rating.
We need to make a 2D matrix with users as rows and movies as columns. Cells should be 0 or 1 depending on if the movie has been rated by the user. 
To find similar users, LSH should be used to find candidates to be similar. For the candidates, Jaccard similarity can be used to find most similar users. 

## Step by Step
1. Load in data
2. Reform data
3. Create signatures for users
4. Split signatures into bands
5. Put similar users in same bucket
6. Compute Jaccard similarity on similar users


LSH(locality sensitive hashing) is a technique for quickly finding similar items in large datasets without comparing everything. Similar items are likely to get the same hash value. LSH uses special hash functions that cluster similar things together.
Each user is represented as the set of movies that they rated. minhash signature is its compressed representation.
Signatures are split up into 'bands', each band is split up into rows. If two users have identical rows in at least one band, they might be similar users.

for these 'might be similar' users, compute jaccard similarity to find pairs of most similar users.

### 1. Load in data
Check if file exists, check if file has format as expected, parse columns. get number of users and number of movies

In [ ]:
# load user_movie_rating.npy and parse columns

import numpy as np
import os

# check if file exists
user_movie_rating_path = "user_movie_rating.npy"
if not os.path.exists(user_movie_rating_path):
    raise FileNotFoundError(f"{user_movie_rating_path} not found in current directory")

# load data and raise error if data is not as expected
data = np.load(user_movie_rating_path, mmap_mode='r') # mmap_mode='r' for large files
if data.ndim != 2 or data.shape[1] != 3:
    raise ValueError("Expected a 2D array with 3 columns: user_id, movie_id, rating")

# parse columns
users = data[:, 0].astype(int)
movies = data[:, 1].astype(int)
ratings = data[:, 2].astype(int)

#! at the end the specific rating does not matter for building the user-item matrix

n_users = users.max()
n_movies = movies.max()

print(f"Loaded {data.shape[0]} interactions")
print(f"Users: {n_users} ")
print(f"Movies: {n_movies} ")

# example: show first 10 raw rows and their mapped indices
print("First 10 raw rows (user_id, movie_id, rating):")
print(data[:10])



Loaded 65225506 interactions
Users: 103703 
Movies: 17770 
First 10 raw rows (user_id, movie_id, rating):
[[  1  30   3]
 [  1 157   3]
 [  1 173   4]
 [  1 175   5]
 [  1 191   2]
 [  1 197   3]
 [  1 241   3]
 [  1 295   4]
 [  1 299   3]
 [  1 329   4]]


## 2. Reform data
Data should be stores in a sparse scipy matrix. In below block, different formats are displayed and best practises are shown. We use COO to create sparse matrix. Then we transform it to CSR to do further calculations on. create list of arrays, each array contains the movie indices rated by that user

# what sparse matrix storage scheme to use
| Format | Matrix × Vector | Get Item | Fancy Get | Set Item | Fancy Set | Solvers | Notes |
|--------|------------------|----------|-----------|----------|-----------|---------|--------|
| **DIA** | sparsetools | . | . | . | . | iterative | has data array, specialized |
| **LIL** | via CSR | yes | yes | yes | yes | iterative | arithmetics via CSR, incremental construction |
| **DOK** | python | yes | one axis only | yes | yes | iterative | O(1) item access, incremental construction |
| **COO** | sparsetools | . | . | . | . | iterative | has data array, facilitates fast conversion |
| **CSR** | sparsetools | yes | yes | slow | . | any | has data array, fast row-wise ops |
| **CSC** | sparsetools | yes | yes | slow | . | any | has data array, fast column-wise ops |
| **BSR** | sparsetools | . | . | . | . | specialized | has data array, specialized |

For LSH we want fast row access because we create hash values from users(set of movies they rated) CSR is good for this
For the cadidates to be similar, we need to calculate jaccard similarity. we need to look at the intersections of the two movie sets of the users. 

To build the sparse matrix COO LIL and DOK are mostly used. 


In [ ]:
# create a sparse user-item rating matrix

from scipy.sparse import coo_matrix

#scipy uses ID's starting from 0
users -= 1
movies -= 1

data_values = np.ones_like(users, dtype=np.uint8) # rating presence indicator, ratings is ignored
coo = coo_matrix((data_values, (users, movies)), shape=(n_users, n_movies), dtype=bool)
print(f"Sparse rating matrix shape: {coo.shape}, nnz={coo.nnz}")
del users, movies, ratings, data
csr = coo.tocsr()
del coo

#print some csr data
print(f"CSR matrix shape: {csr.shape}, nnz={csr.nnz}")
print(f"CSR matrix data sample (first 10 entries): {csr.data[:40]}")

# create list of arrays, each array contains the movie indices rated by that user
user_movie_lists = [
    csr.indices[csr.indptr[i]:csr.indptr[i+1]]
    for i in range(n_users)
]

Sparse rating matrix shape: (103703, 17770), nnz=65225506
CSR matrix shape: (103703, 17770), nnz=65225506
CSR matrix data sample (first 10 entries): [ True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True]


## 3. Create signatures for users
Set number of has functions/permutations, bands and rows. Create random permutations, create signature matrix; k signatures per user.

In [ ]:
from tqdm import tqdm #! remove eventually because we can only use numpy and scipy libraries

k = 120 # number of hash functions (permutations)
bands = 30
rows = k // bands # rows is now 4

# create k random permutations of movie indices
permutations = np.array([
    np.random.permutation(n_movies)
    for a in range(k)
], dtype=np.int32)

# initialize signature matrix
signatures = np.empty(shape=(n_users, k), dtype=np.int32)

# loop over each permutation and compute minhash signatures
for j in tqdm(range(k)): #! tqdm to show progress bar. remove in final version
    perm = permutations[j]
    # compute: min perm[movie] for each user
    signatures[:, j] = np.array([
        perm[movies].min() # mishash value for this user and this permutation
        for movies in user_movie_lists
    ])

print(signatures[0:5, :10])  # print first 5 users first 10 signature values
print(signatures.shape)


100%|██████████| 120/120 [00:46<00:00,  2.59it/s]

[[ 48  25   0  59  38  28  54   3 109  13]
 [ 17  25  18   3  19   0  27   3  22  13]
 [  8  25   0  93  85   0  12  16   5  13]
 [ 47  25  18  33  19 151  16   5  91  66]
 [ 17  25  52  93   1  89  16  16  22  24]]


## 4. Split signatures into bands


In [25]:
banded_signatures = signatures.reshape(n_users, bands, rows)
print(banded_signatures.shape)
print("User 0, band 0:", banded_signatures[0, 0, :])
print("User 0, band 1:", banded_signatures[0, 1, :])

(103703, 30, 4)
User 0, band 0: [48 25  0 59]
User 0, band 1: [38 28 54  3]


## 5. put similar users in same bucket
assign each band to a bucket